<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/08_Determinacion_de_g_y_comparacion_de_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 08 — Determinación de g y comparación de modelos anidados**Laboratorio 1 · Clase 8****Objetivos.**1. Ajustar $x(t) = x_0 + v_0 t + \\tfrac{1}{2}a t^2$ con pesos y extraer $g$ **con su incerteza**.2. Decidir con criterio si un término adicional del modelo **está justificado por los datos**.3. Reconocer un error sistemático en el resultado final y saber qué hacer (y qué no).**Requisitos previos:** Colabs 05 a 07.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitfrom scipy import statsnp.random.seed(20260930)def chi2_reducido(y, y_mod, sigma, p, verbose=True):    y, y_mod, sigma = map(lambda v: np.asarray(v, float), (y, y_mod, sigma))    chi2 = np.sum(((y - y_mod)/sigma)**2)    nu = len(y) - p    if verbose:        print(f"χ² = {chi2:.2f}  ν = {nu}  χ²_ν = {chi2/nu:.2f}  "              f"p = {stats.chi2.sf(chi2, nu):.3f}")    return chi2/nu

---## 1. El modelo y el ajustePara un móvil con aceleración constante,$$ x(t) = x_0 + v_0 t + \\tfrac{1}{2} a t^2 $$Es un modelo **no lineal en la variable** $t$ pero **lineal en los parámetros** $(x_0, v_0, a)$: poreso el ajuste tiene solución cerrada y `curve_fit` converge siempre, sin necesidad de semillas.

In [ ]:
# --- DATOS DE EJEMPLO: plano inclinado, θ = 12,0° (reemplazar por los propios) ---theta, dtheta = np.radians(25.0), np.radians(0.2)a_real = 9.81*np.sin(theta) * 0.97          # 3 % menos por rozamiento: efecto sistemático realt = np.arange(0, 1.2, 1/60)                  # cámara a 60 fpssigma_x = 0.003                              # m, del error de identificación en Trackerx = 0.02 + 0.05*t + 0.5*a_real*t**2 + np.random.normal(0, sigma_x, len(t))err_x = np.full_like(t, sigma_x)# --------------------------------------------------------------------------------def parabola(t, x0, v0, a):    return x0 + v0*t + 0.5*a*t**2popt, pcov = curve_fit(parabola, t, x, sigma=err_x, absolute_sigma=True)perr = np.sqrt(np.diag(pcov))for nombre, val, e, u in zip(['x0', 'v0', 'a'], popt, perr, ['m', 'm/s', 'm/s²']):    print(f"{nombre} = {val:9.5f} ± {e:.5f} {u}")c2r = chi2_reducido(x, parabola(t, *popt), err_x, 3)

---## 2. De la aceleración a $g$: propagación con dos fuentesEn el plano inclinado sin rozamiento, $a = g\\sin\\theta$, de donde$$ g = \\frac{a}{\\sin\\theta}, \\qquad   \\left(\\frac{\\sigma_g}{g}\\right)^2 = \\left(\\frac{\\sigma_a}{a}\\right)^2 +   \\left(\\frac{\\sigma_\\theta}{\\tan\\theta}\\right)^2 $$Nótese que el ángulo entra dividido por $\\tan\\theta$: **a ángulos chicos, el error del ángulodomina todo**. Es una decisión de diseño experimental, no un detalle de cuentas.

In [ ]:
a_fit, da_fit = popt[2], perr[2]g = a_fit/np.sin(theta)dg = g*np.sqrt((da_fit/a_fit)**2 + (dtheta/np.tan(theta))**2)contrib_a = (da_fit/a_fit)**2contrib_th = (dtheta/np.tan(theta))**2print(f"g = {g:.3f} ± {dg:.3f} m/s²")print(f"\n  contribución de a : {100*contrib_a/(contrib_a+contrib_th):5.1f} %")print(f"  contribución de θ : {100*contrib_th/(contrib_a+contrib_th):5.1f} %")print(f"\nDiscrepancia con 9,81:  z = {abs(g-9.81)/dg:.1f}")

> **Ejercicio 8.1.** Recalculá con $\\theta = 30°$ y el mismo $\\sigma_\\theta$. ¿Cuánto baja la> contribución del ángulo? ¿Conviene entonces trabajar siempre con ángulos grandes? (Pensá qué pasa> con la duración del recorrido y con la cantidad de puntos que llegás a medir.)

---## 3. ¿Hace falta un término más? Modelos anidadosEn el resultado anterior $g$ salió bajo, con una discrepancia cercana a $4\sigma$. Sospecharazonable: hay rozamiento, y el modelo de aceleración constante no lo contempla.Podríamos agregar un término. Pero acá aparece una trampa lógica:> **Agregar parámetros siempre baja el $\\chi^2$.** Siempre. Un polinomio de grado $N-1$ pasa> exactamente por $N$ puntos y da $\\chi^2 = 0$, y no describe absolutamente nada.Lo que hay que mirar es el **$\\chi^2$ reducido**, que divide por $\\nu = N - p$ y por lo tantopenaliza el parámetro extra, junto con **la estructura de los residuos**. Si el término nuevo nomejora $\\chi^2_\\nu$ y los residuos ya eran ruido, el término sobra.

In [ ]:
def parabola_mas_cubico(t, x0, v0, a, c):    return x0 + v0*t + 0.5*a*t**2 + c*t**3p3, c3 = curve_fit(parabola_mas_cubico, t, x, sigma=err_x, absolute_sigma=True)e3 = np.sqrt(np.diag(c3))chi2_2 = np.sum(((x - parabola(t, *popt))/err_x)**2)chi2_3 = np.sum(((x - parabola_mas_cubico(t, *p3))/err_x)**2)N = len(t)print(f"{'modelo':<26}{'p':>3}{'χ²':>12}{'χ²_ν':>10}")print("-"*52)print(f"{'x0 + v0 t + a t²/2':<26}{3:>3}{chi2_2:>12.1f}{chi2_2/(N-3):>10.3f}")print(f"{'... + c t³':<26}{4:>3}{chi2_3:>12.1f}{chi2_3/(N-4):>10.3f}")print(f"\nc = {p3[3]:.5f} ± {e3[3]:.5f}   ->  |c|/σ_c = {abs(p3[3]/e3[3]):.2f}")

Leé las dos últimas cifras. El $\\chi^2$ bajó (tenía que bajar), pero el $\\chi^2_\\nu$ prácticamenteno cambió, y el parámetro nuevo es compatible con cero (su valor es menor que dos veces su propioerror). Conclusión: **el término cúbico no está justificado por estos datos**.Criterio operativo para el curso: un parámetro adicional se justifica si (a) mejoraapreciablemente el $\\chi^2_\\nu$, (b) es distinto de cero con significancia razonable($|c| > 2\\sigma_c$, mejor $3\\sigma_c$) y (c) hay una razón física para incluirlo. Las tres, no una.*(Existen criterios formales —test F para modelos anidados, AIC/BIC para no anidados— pero quedancomo contenido avanzado optativo. Para Laboratorio 1, $\\chi^2_\\nu$ + residuos + significancia delparámetro alcanza y sobra.)*

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 6), sharex=True,                             gridspec_kw={'height_ratios': [2, 1]})a1.errorbar(t[::4], x[::4], yerr=err_x[::4], fmt='o', ms=3, capsize=2, alpha=0.6, label='datos')a1.plot(t, parabola(t, *popt), 'crimson', lw=1.6, label='3 parámetros')a1.plot(t, parabola_mas_cubico(t, *p3), 'navy', lw=1.2, ls='--', label='4 parámetros')a1.set_ylabel('$x$ [m]'); a1.grid(alpha=0.3); a1.legend()a1.set_title('Los dos modelos son visualmente indistinguibles')a2.plot(t, (x - parabola(t, *popt))*1000, '.', ms=3, color='crimson', label='3 par.')a2.plot(t, (x - parabola_mas_cubico(t, *p3))*1000, '.', ms=3, color='navy', label='4 par.')a2.axhline(0, color='k', lw=1)a2.set_xlabel('$t$ [s]'); a2.set_ylabel('residuo [mm]'); a2.grid(alpha=0.3); a2.legend(ncol=2)fig.subplots_adjust(hspace=0.08); plt.show()

---## 4. Cuando el problema es sistemáticoVolvamos al resultado: $g$ dio bajo, y con una discrepancia de varias $\\sigma$. El $\\chi^2_\\nu$ delajuste, sin embargo, es razonable y los residuos no tienen estructura. **El modelo describe bien losdatos; los datos describen mal la gravedad.**Ése es el diagnóstico de un error sistemático, y tiene tres salidas posibles:1. **Eliminarlo** — reducir el rozamiento, mejorar la alineación, recalibrar el ángulo.2. **Modelarlo** — incluir el rozamiento en la ecuación de movimiento con un parámetro físico   (no un término polinómico arbitrario) y ajustar.3. **Acotarlo y declararlo** — estimar su magnitud y reportarlo como incerteza sistemática, separada   de la estadística: $g = (9{,}52 \\pm 0{,}05_{\\text{est}} \\pm 0{,}20_{\\text{sist}})$ m/s².Lo que **no** es una salida: medir más veces. La incerteza estadística baja, la sistemática no semueve, y el resultado se vuelve más preciso y más falso — exactamente lo que vimos en el Colab 04.

In [ ]:
# Opción 2: modelar el rozamiento con un coeficiente físico#   m a = m g senθ - μ m g cosθ   =>   a = g (senθ - μ cosθ)def parabola_con_roce(t, x0, v0, mu, g_fijo=9.81):    a = g_fijo*(np.sin(theta) - mu*np.cos(theta))    return x0 + v0*t + 0.5*a*t**2pr, cr = curve_fit(parabola_con_roce, t, x, sigma=err_x, absolute_sigma=True, p0=[0.02, 0.05, 0.01])er = np.sqrt(np.diag(cr))print(f"μ (coef. de rozamiento) = {pr[2]:.4f} ± {er[2]:.4f}")print("χ²_ν de este modelo:", end=" ")chi2_reducido(x, parabola_con_roce(t, *pr), err_x, 3)

Ahora el parámetro extra **sí** tiene sentido físico, es distinto de cero con significancia, y suvalor es del orden esperable para un contacto metal-metal. Ésa es la diferencia entre agregar untérmino y modelar un efecto.

---## 5. Ejercicios**8.2.** Con tus datos, determiná $g$ por caída libre y por plano inclinado. Reportá ambos con suincerteza y evaluá compatibilidad con `compatibilidad()` del Colab 03.**8.3.** Si te da $z > 3$, escribí las tres hipótesis más probables ordenadas por verosimilitud, yproponé una medición que las distinga.**8.4.** Ajustá el mismo conjunto usando solo la primera mitad de los datos y solo la segunda.¿Coinciden las aceleraciones? Una diferencia significativa entre ambas mitades es evidencia directade que la aceleración **no** es constante.**8.5.** *(diseño)* ¿Cuántos puntos y en qué rango de $t$ minimizarían $\\sigma_a$ para un tiempototal de adquisición fijo? Probalo numéricamente recortando la ventana temporal y viendo cómo cambia`perr[2]`.